# 2. Traffic Forecasting

Forecasts passenger and freight traffic with Prophet, **and backtests each forecast** against a
held-out period -- the original notebook never measured accuracy at all.

In [ ]:
import sys

sys.path.insert(0, '../src')

from pathlib import Path

import pandas as pd

from international_airlines.forecasting import backtest, fit_forecast, to_prophet_frame

RESULTS_DIR = Path('../results')
df = pd.read_csv(RESULTS_DIR / 'processed_city_pairs.csv', parse_dates=['Date'])

monthly = df.groupby('Date', as_index=False)[['Passengers_Total', 'Freight_Total_(tonnes)']].sum()
monthly.tail()

In [ ]:
passenger_ts = to_prophet_frame(monthly, 'Date', 'Passengers_Total')

print('Backtest (last 12 months held out):', backtest(passenger_ts, holdout_periods=12))

model, forecast = fit_forecast(passenger_ts, periods=12)
fig = model.plot(forecast)
fig2 = model.plot_components(forecast)

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(RESULTS_DIR / 'passenger_forecast.csv', index=False)

In [ ]:
freight_ts = to_prophet_frame(monthly, 'Date', 'Freight_Total_(tonnes)')

print('Backtest (last 12 months held out):', backtest(freight_ts, holdout_periods=12))

model_freight, forecast_freight = fit_forecast(freight_ts, periods=12)
fig = model_freight.plot(forecast_freight)

forecast_freight[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(RESULTS_DIR / 'freight_forecast.csv', index=False)